# Tech Challenge Fase 2  
## Notebook 01.1 — Bronze Alunos

### Responsabilidade do notebook

Este notebook é responsável pela ingestão dos arquivos de **alunos** da avaliação de alfabetização, recebidos em formato CSV, para a camada **Bronze** do Data Lake.

A camada Bronze preserva os dados o mais próximo possível da origem, adicionando apenas metadados técnicos de rastreabilidade.

---

### Entrada

```text
raw/alunos/
├── TS_ALUNO_2023.csv
├── TS_ALUNO_2024.csv
└── TS_ALUNO_2025.csv
```

### Saída

```text
bronze/alunos/
├── ano=2023/
├── ano=2024/
└── ano=2025/
```

### Dependências

- `00_setup_ambiente`
- `01_bronze_orquestrador`
- `config/config.json`
- `config/bronze_metadata`

### Próximo notebook

```text
01_2_bronze_estados
```

## 1. Contexto na Arquitetura Medalhão

Este notebook atua exclusivamente na camada **Bronze**.

```text
Raw
 ↓
Bronze  ← você está aqui
 ↓
Silver
 ↓
Gold
```

A Bronze não aplica regras de negócio, padronização semântica ou tratamento analítico.  
O objetivo principal é garantir **rastreabilidade**, **auditoria** e **possibilidade de reprocessamento**.

## 2. Importação das bibliotecas

Nesta etapa importamos bibliotecas utilizadas para:

- leitura do arquivo de configuração;
- manipulação de datas;
- criação de schemas explícitos;
- uso de funções Spark;
- geração de logs operacionais.

O uso de schema explícito nos logs evita erros de inferência comuns no Databricks Serverless.

In [0]:
import json
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType
)

## 3. Leitura do arquivo de configuração

O projeto utiliza um arquivo central `config.json`, gerado no notebook de setup.

Essa abordagem evita caminhos fixos espalhados pelos notebooks e facilita mudanças futuras de ambiente, bucket, volume ou estrutura de diretórios.

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(dbutils.fs.head(CONFIG_FILE_PATH))

BASE_PATH = config["environment"]["base_path"]
RAW_PATH = config["paths"]["raw_path"]
BRONZE_PATH = config["paths"]["bronze_path"]
LOG_PATH = config["paths"]["log_path"]
CONFIG_PATH = config["paths"]["config_path"]
EXECUTION_DATE = config["project"]["execution_date"]

print("BASE_PATH:", BASE_PATH)
print("RAW_PATH:", RAW_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("LOG_PATH:", LOG_PATH)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Leitura dos metadados Bronze

O notebook planejador criou a tabela `bronze_metadata`, que contém as informações necessárias para processar cada arquivo.

Nesta etapa, filtramos apenas os registros referentes ao dataset `alunos`.

Essa estratégia torna o pipeline orientado por metadados e evita duplicação de código.

In [0]:
metadata_path = f"{CONFIG_PATH}/bronze_metadata"

df_metadata = spark.read.parquet(metadata_path)

df_metadata_alunos = (
    df_metadata
    .filter(F.col("dataset") == "alunos")
    .orderBy("ano")
)

display(df_metadata_alunos)

## 5. Validação dos metadados de alunos

Antes de iniciar a ingestão, validamos se a tabela de metadados contém exatamente os arquivos esperados para os anos de 2023, 2024 e 2025.

Essa validação evita execução parcial do pipeline.

In [0]:
anos_esperados = [2023, 2024, 2025]

anos_metadata = [
    row["ano"] for row in df_metadata_alunos.select("ano").distinct().collect()
]

anos_faltantes = sorted(list(set(anos_esperados) - set(anos_metadata)))

if anos_faltantes:
    raise Exception(f"Metadados ausentes para os anos: {anos_faltantes}")
else:
    print("Metadados de alunos encontrados para todos os anos esperados.")

## 6. Função de leitura dos arquivos CSV

A base de alunos é recebida em formato CSV com separador `;`.

A função abaixo lê cada arquivo conforme os metadados registrados.

Para a camada Bronze, utilizamos `inferSchema=True`, pois o objetivo é preservar a estrutura original inferida a partir dos arquivos recebidos.

Também aplicamos a codificação `ISO-8859-1`, compatível com os arquivos analisados anteriormente.

In [0]:
def read_csv_bronze(file_path: str, separator: str, encoding: str):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .option("sep", separator)
        .option("encoding", encoding)
        .csv(file_path)
    )

    return df

## 7. Função de enriquecimento técnico da Bronze

A Bronze deve preservar os dados originais, mas pode receber metadados técnicos.

Esses campos permitem responder perguntas como:

- De qual arquivo esse registro veio?
- Qual dataset foi processado?
- Qual ano de referência foi carregado?
- Quando a ingestão ocorreu?
- Qual etapa do pipeline gerou esse dado?

In [0]:
def add_bronze_metadata(df, dataset, source_file, source_format, ano_referencia):
    df_bronze = (
        df
        .withColumn("_dataset", F.lit(dataset))
        .withColumn("_source_file", F.lit(source_file))
        .withColumn("_source_format", F.lit(source_format))
        .withColumn("_ano_referencia", F.lit(int(ano_referencia)))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_execution_date", F.lit(EXECUTION_DATE))
        .withColumn("_pipeline_step", F.lit("bronze_alunos"))
    )

    return df_bronze

## 8. Execução da ingestão Bronze — Alunos

Nesta etapa o notebook percorre os metadados dos arquivos de alunos, lê cada CSV, adiciona os metadados técnicos e grava o resultado em Parquet na camada Bronze.

A saída é organizada por ano:

```text
bronze/alunos/ano=2023
bronze/alunos/ano=2024
bronze/alunos/ano=2025
```

### Observação técnica

Para evitar o problema de muitos arquivos pequenos em datasets menores, utilizamos `coalesce(1)` em 2023 e 2024 quando aplicável.  
Para bases maiores, essa decisão pode ser revisada, mas para a estrutura atual ela favorece simplicidade e legibilidade no Data Lake.

In [0]:
execution_logs = []

rows_metadata = df_metadata_alunos.collect()

for row in rows_metadata:
    dataset = row["dataset"]
    ano = int(row["ano"])
    file_name = row["file_name"]
    raw_path = row["raw_path"]
    bronze_output_path = row["bronze_path"]
    source_format = row["source_format"]
    separator = row["separator"]
    encoding = row["encoding"] if row["encoding"] else "ISO-8859-1"

    start_time = datetime.now()

    print("=" * 100)
    print(f"Iniciando ingestão Bronze — Dataset: {dataset} | Ano: {ano}")
    print(f"Arquivo origem: {raw_path}")
    print(f"Destino Bronze: {bronze_output_path}")

    try:
        df_raw = read_csv_bronze(
            file_path=raw_path,
            separator=separator,
            encoding=encoding
        )

        df_bronze = add_bronze_metadata(
            df=df_raw,
            dataset=dataset,
            source_file=raw_path,
            source_format=source_format,
            ano_referencia=ano
        )

        records_read = df_bronze.count()
        columns_count = len(df_bronze.columns)

        (
            df_bronze
            .coalesce(1)
            .write
            .mode("overwrite")
            .format("parquet")
            .option("compression", "snappy")
            .save(bronze_output_path)
        )

        end_time = datetime.now()

        execution_logs.append({
            "dataset": str(dataset),
            "ano": int(ano),
            "file_name": str(file_name),
            "source_path": str(raw_path),
            "target_path": str(bronze_output_path),
            "status": "SUCCESS",
            "records_read": int(records_read),
            "columns_count": int(columns_count),
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error_message": ""
        })

        print(f"Ingestão concluída com sucesso para {dataset} {ano}")
        print(f"Registros lidos: {records_read}")
        print(f"Colunas: {columns_count}")

    except Exception as e:
        end_time = datetime.now()

        execution_logs.append({
            "dataset": str(dataset),
            "ano": int(ano),
            "file_name": str(file_name),
            "source_path": str(raw_path),
            "target_path": str(bronze_output_path),
            "status": "FAILED",
            "records_read": 0,
            "columns_count": 0,
            "start_time": start_time.isoformat(),
            "end_time": end_time.isoformat(),
            "error_message": str(e)
        })

        print(f"Erro na ingestão do arquivo {file_name}: {e}")

## 9. Criação do log de execução

Após a ingestão, os resultados de cada carga são convertidos em um DataFrame Spark com schema explícito.

Esse log registra:

- dataset;
- ano;
- arquivo de origem;
- destino;
- status;
- quantidade de registros;
- quantidade de colunas;
- horários de início e fim;
- mensagem de erro, se existir.

In [0]:
schema_execution_logs = StructType([
    StructField("dataset", StringType(), True),
    StructField("ano", IntegerType(), True),
    StructField("file_name", StringType(), True),
    StructField("source_path", StringType(), True),
    StructField("target_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_read", LongType(), True),
    StructField("columns_count", IntegerType(), True),
    StructField("start_time", StringType(), True),
    StructField("end_time", StringType(), True),
    StructField("error_message", StringType(), True)
])

df_execution_logs = spark.createDataFrame(
    execution_logs,
    schema=schema_execution_logs
)

display(df_execution_logs.orderBy("ano"))

## 10. Persistência do log de execução

Os logs são armazenados na camada `logs/pipeline_execution/bronze`, permitindo auditoria posterior da execução.

Essa etapa é importante para observabilidade do pipeline.

In [0]:
log_output_path = f"{LOG_PATH}/pipeline_execution/bronze/alunos_execution_date={EXECUTION_DATE}"

(
    df_execution_logs
    .coalesce(1)
    .write
    .mode("overwrite")
    .format("parquet")
    .option("compression", "snappy")
    .save(log_output_path)
)

print("Log de execução salvo em:")
print(log_output_path)

## 11. Validação da camada Bronze gerada

Após a gravação, lemos os arquivos Parquet gerados para confirmar se a ingestão foi concluída corretamente.

A validação apresenta uma amostra dos dados e confirma a existência das pastas por ano.

In [0]:
for row in rows_metadata:
    ano = int(row["ano"])
    bronze_output_path = row["bronze_path"]

    print("=" * 100)
    print(f"Validação Bronze — alunos | ano={ano}")
    print(bronze_output_path)

    try:
        df_validacao = spark.read.parquet(bronze_output_path)
        print("Colunas:", len(df_validacao.columns))
        display(df_validacao.limit(5))
    except Exception as e:
        print(f"Erro ao validar Bronze alunos ano={ano}: {e}")

## 12. Checklist final do notebook

Esta etapa apresenta um resumo da execução para facilitar a leitura operacional do notebook.

Caso exista alguma falha, o notebook retorna uma exceção para impedir que os próximos passos sejam executados sobre uma Bronze incompleta.

In [0]:
falhas = df_execution_logs.filter(F.col("status") == "FAILED").count()

if falhas > 0:
    display(df_execution_logs.filter(F.col("status") == "FAILED"))
    raise Exception(f"Foram encontradas {falhas} falhas na ingestão Bronze de alunos.")
else:
    print("Checklist final concluído com sucesso.")
    print("Todos os arquivos de alunos foram ingeridos na camada Bronze.")

## Resultado esperado

Ao final deste notebook, espera-se que a camada Bronze de alunos esteja criada:

```text
bronze/alunos/
├── ano=2023/
├── ano=2024/
└── ano=2025/
```

Cada pasta deve conter arquivos Parquet com os dados originais acrescidos de metadados técnicos.

---

## Próximo notebook

```text
01_2_bronze_estados
```